In [ ]:
import pandas as pd
import numpy as np
import re
import tensorflow as tf
from tensorflow.keras import layers, models, callbacks
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from scipy.stats import pearsonr

In [ ]:
def prepare_resnet_data(file_path):
    df = pd.read_csv(file_path)

    # Parse Ticker for Strike and Expiry (e.g., AARTIIND28NOV24490PE.NFO)
    def parse_ticker(ticker):
        match = re.search(r"(\d{2}[A-Z]{3}\d{2})(\d+)[CP]E", str(ticker))
        return match.groups() if match else (None, None)

    df[['Expiry_Str', 'Strike']] = df['Ticker'].apply(lambda x: pd.Series(parse_ticker(x)))
    df = df.dropna(subset=['Strike', 'Close'])
    df['Strike'] = df['Strike'].astype(float)

    # Core ResNet Standardizations: S/K and V/K
    df['S_K'] = df['Open'] / df['Strike']
    df['V_K'] = df['Close'] / df['Strike'] # Target

    # Time to Expiry (T)
    df['Date_dt'] = pd.to_datetime(df['Date'], dayfirst=True)
    df['Expiry_dt'] = pd.to_datetime(df['Expiry_Str'], format='%d%b%y')
    df['T'] = (df['Expiry_dt'] - df['Date_dt']).dt.days / 365.0
    df = df[df['T'] > 0]

    # Features as used in the study: S/K, T, and Volume/OI as liquidity proxies
    features = ['S_K', 'T', 'Volume', 'Open Interest']
    X = df[features].values
    y = df['V_K'].values

    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)

    # Reshape for Conv1D: (samples, features, 1)
    X_reshaped = X_scaled.reshape((X_scaled.shape[0], X_scaled.shape[1], 1))

    return train_test_split(X_reshaped, y, test_size=0.2, random_state=42)

In [ ]:
def residual_block(x, filters):
    shortcut = x
    # Three 1D Convolution layers per block
    x = layers.Conv1D(filters, 3, padding='same', activation='relu')(x)
    x = layers.Conv1D(filters, 3, padding='same', activation='relu')(x)
    x = layers.Conv1D(filters, 3, padding='same')(x) # Linear before addition

    # Adjust shortcut dimension if necessary
    if shortcut.shape[-1] != filters:
        shortcut = layers.Conv1D(filters, 1, padding='same')(shortcut)

    x = layers.Add()([x, shortcut])
    x = layers.Activation('relu')(x)
    return x

In [ ]:
def build_resnet_24(input_shape):
    inputs = layers.Input(shape=input_shape)
    x = inputs

    # 4 Block A units (64 filters)
    for _ in range(4):
        x = residual_block(x, 64)
    # 4 Block B units (128 filters)
    for _ in range(4):
        x = residual_block(x, 128)

    # Global Average Pooling + ELU Activation as per Paper Sect 3.2
    x = layers.GlobalAveragePooling1D()(x)
    x = layers.Dense(64, activation='elu')(x)
    outputs = layers.Dense(1, activation='linear')(x)

    model = models.Model(inputs, outputs)
    model.compile(optimizer=tf.keras.optimizers.Adam(0.001), loss='mse')
    return model

In [ ]:
X_train, X_test, y_train, y_test = prepare_resnet_data('NSE_FNO_DATA_2024-10-21.CSV')
model = build_resnet_24(X_train.shape[1:])

# Training
model.fit(X_train, y_train, epochs=10, batch_size=256, verbose=0)

# Predictions
y_pred = model.predict(X_test).flatten()

# Metrics
mse = np.mean((y_test - y_pred)**2)
pcc, _ = pearsonr(y_test, y_pred)

print(f"--- ResNet (24-Layer) Results ---")
print(f"MSE: {mse:.8f}")
print(f"PCC: {pcc:.4f}")